In [0]:
%pip install rank-bm25 sentence-transformers

In [0]:
dbutils.library.restartPython()

In [0]:
import re, unicodedata, numpy as np
from rank_bm25 import BM25Okapi

docs = spark.table("workspace.forecast_debug.rag_documents").collect()
print("doc_id disponibles :", [d["doc_id"] for d in docs])

chunks = []
for d in docs:
    parts = re.split(r"\n(?=#{1,3} )", d["content"])
    for i, p in enumerate(parts):
        p = p.strip()
        if len(p) > 40:
            chunks.append({"doc_id": d["doc_id"], "title": d["title"],
                           "chunk_id": f'{d["doc_id"]}#{i}',
                           "text": f'{d["title"]}\n{p}'})
print(len(chunks), "passages")
print(chunks[0]["text"][:300])

In [0]:
def tokenize(s):
    s = unicodedata.normalize("NFD", s.lower())
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")
    return re.findall(r"[a-z0-9_]+", s)

bm25 = BM25Okapi([tokenize(c["text"]) for c in chunks])

def rank_bm25(query):
    scores = bm25.get_scores(tokenize(query))
    return list(np.argsort(-scores))

# test rapide
for i in rank_bm25("snapshot obsolète 24 h")[:3]:
    print(chunks[i]["chunk_id"], "|", chunks[i]["title"])

In [0]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
emb = model.encode([c["text"] for c in chunks], normalize_embeddings=True)

def rank_dense(query):
    q = model.encode([query], normalize_embeddings=True)[0]
    return list(np.argsort(-(emb @ q)))

for i in rank_dense("les données sont trop vieilles")[:3]:
    print(chunks[i]["chunk_id"], "|", chunks[i]["title"])

In [0]:
def rrf(rankings, k=60):
    score = {}
    for ranking in rankings:
        for pos, idx in enumerate(ranking):
            score[idx] = score.get(idx, 0) + 1 / (k + pos + 1)
    return sorted(score, key=score.get, reverse=True)

def search_knowledge(query, k=3):
    order = rrf([rank_bm25(query), rank_dense(query)])[:k]
    return [{"doc_id": chunks[i]["doc_id"], "title": chunks[i]["title"],
             "passage": chunks[i]["text"]} for i in order]

import json
print(json.dumps(search_knowledge("un écart de 40 % avec un snapshot de 50 heures", 3),
                 indent=2, ensure_ascii=False)[:1500])

In [0]:
tests = [  # (requête, mot-clé attendu dans doc_id)
    ("à partir de quand un snapshot est obsolète", "fresh"),
    ("les données sont trop vieilles", "fresh"),
    ("quel écart est considéré normal", "toler"),
    ("la promo n'était pas synchronisée entre les deux systèmes", "promo"),
    ("comment est calculée sales_7d", "feature"),
    ("le modèle a été déployé en version V3 d'un seul côté", "model"),
    ("par quoi commencer pour investiguer un écart", "runbook"),
    # requêtes maladroites, éloignées des documents :
    ("pourquoi le batch est en retard par rapport au temps réel", "fresh"),
    ("est-ce grave si les chiffres diffèrent un peu", "toler"),
    ("un des deux calculs utilise une autre réglage", "config"),
]

def recall_at_3(ranker):
    ok = 0
    for q, kw in tests:
        top = ranker(q)[:3]
        ok += any(kw in chunks[i]["doc_id"].lower() for i in top)
    return ok, len(tests)

print("BM25    :", recall_at_3(rank_bm25))
print("Dense   :", recall_at_3(rank_dense))
print("Hybride :", recall_at_3(lambda q: rrf([rank_bm25(q), rank_dense(q)])))